# Spiking PointNet + Q-SDE: Results Analysis

This notebook compares Spiking PointNet (baseline, direct encoding) against
the Q-SDE variant on ModelNet40 point cloud classification.

**Before running:** ensure you have completed at least one training run and
that `results/metrics_*.csv` files exist.

In [ ]:
import sys
from pathlib import Path

# Add src to path
repo_root = Path('.').resolve().parent
sys.path.insert(0, str(repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RESULTS_DIR = repo_root / 'results'
print(f"Results directory: {RESULTS_DIR}")
csv_files = list(RESULTS_DIR.glob('metrics_*.csv'))
print(f"Found {len(csv_files)} metrics CSV files:")
for f in csv_files:
    print(f"  {f.name}")

## 1. Load Results CSVs

In [ ]:
# Load all available metrics CSVs
dfs = {}
for csv_path in sorted(csv_files):
    run_name = csv_path.stem.replace('metrics_', '')
    df = pd.read_csv(csv_path)
    dfs[run_name] = df
    print(f"  {run_name}: {len(df)} epochs, best val OA = {df['val_oa'].max()*100:.2f}%")

if not dfs:
    print("\nNo metrics CSVs found. Train a model first:")
    print("  python scripts/train.py --config configs/baseline.yaml")
else:
    print(f"\nLoaded {len(dfs)} run(s).")

## 2. Training Loss and OA Curves

In [ ]:
if not dfs:
    print("No data to plot.")
else:
    colors = plt.cm.tab10.colors
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Training Curves: Baseline vs Q-SDE', fontsize=14, fontweight='bold')

    for i, (run_name, df) in enumerate(dfs.items()):
        color = colors[i % len(colors)]
        label = run_name

        # Loss
        axes[0].plot(df['epoch'], df['train_loss'], color=color, linewidth=1.5,
                     label=f'{label} (train)', linestyle='-')
        axes[0].plot(df['epoch'], df['val_loss'], color=color, linewidth=1.5,
                     label=f'{label} (val)', linestyle='--')

        # OA
        axes[1].plot(df['epoch'], df['val_oa'] * 100, color=color, linewidth=2,
                     label=f'{label}')
        best_oa = df['val_oa'].max() * 100
        best_ep = df.loc[df['val_oa'].idxmax(), 'epoch']
        axes[1].axhline(best_oa, color=color, linestyle=':', alpha=0.5)
        axes[1].annotate(f'{best_oa:.2f}%',
                         xy=(best_ep, best_oa),
                         xytext=(best_ep + 2, best_oa - 0.3),
                         fontsize=9, color=color)

    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training & Validation Loss')
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Val OA (%)')
    axes[1].set_title('Validation Overall Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    out_path = RESULTS_DIR / 'training_curves_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"Saved to {out_path}")
    plt.show()

## 3. Bar Chart: Final Best OA Comparison

In [ ]:
if not dfs:
    print("No data to plot.")
else:
    # Reference values
    reference = {
        'ANN PointNet\n(reference)': 89.2,
        'Spiking PointNet\n(paper)': 88.61,
    }

    # Our results
    our_results = {name: df['val_oa'].max() * 100 for name, df in dfs.items()}

    # Combine
    all_models = list(reference.keys()) + list(our_results.keys())
    all_oas = list(reference.values()) + list(our_results.values())
    colors_bar = ['#aaaaaa', '#888888'] + [plt.cm.tab10.colors[i] for i in range(len(our_results))]

    fig, ax = plt.subplots(figsize=(max(8, len(all_models) * 2), 5))
    bars = ax.bar(all_models, all_oas, color=colors_bar, edgecolor='white', linewidth=1.5)

    for bar, oa in zip(bars, all_oas):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'{oa:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_ylabel('ModelNet40 OA (%)', fontsize=12)
    ax.set_title('ModelNet40 Overall Accuracy Comparison', fontsize=14, fontweight='bold')
    ymin = min(all_oas) * 0.98
    ymax = max(all_oas) * 1.01
    ax.set_ylim(ymin, ymax)
    ax.axhline(88.61, color='gray', linestyle='--', alpha=0.5, label='Paper baseline (88.61%)')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    out_path = RESULTS_DIR / 'oa_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"Saved to {out_path}")
    plt.show()

## 4. SynOps Energy Comparison

Run energy estimation on a trained model. Requires the checkpoint to be available.

In [ ]:
import torch
from pathlib import Path

repo_root_str = str(repo_root)
checkpoints_dir = repo_root / 'checkpoints'
best_ckpts = list(checkpoints_dir.glob('*_best.pth')) if checkpoints_dir.exists() else []

print(f"Found {len(best_ckpts)} best checkpoints:")
for c in best_ckpts:
    print(f"  {c.name}")

energy_results = []  # will fill in the loop below

In [ ]:
# Compute SynOps for each available checkpoint
if best_ckpts:
    from spk_pointnet.models.spiking_pointnet import SpikingPointNet
    from spk_pointnet.encoding.direct_encoding import direct_encode
    from spk_pointnet.encoding.qsde import qsde_encode
    from spk_pointnet.utils.synops import SynOpsCounter
    from spikingjelly.activation_based import functional

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    for ckpt_path in best_ckpts:
        ckpt = torch.load(str(ckpt_path), map_location=device)
        config = ckpt.get('config', {})
        model_cfg = config.get('model', {})
        enc_cfg = config.get('encoding', {})

        model = SpikingPointNet(
            num_classes=model_cfg.get('num_classes', 40),
            tau=model_cfg.get('tau', 0.25),
            V_th=model_cfg.get('V_th', 0.5),
            dropout=model_cfg.get('dropout', 0.3),
        )
        functional.set_step_mode(model, step_mode='m')
        model.load_state_dict(ckpt['model_state_dict'])
        model = model.to(device)
        model.eval()

        # Generate dummy input
        T_infer = model_cfg.get('T_infer', 4)
        encoding_type = enc_cfg.get('type', 'direct')
        Ns = enc_cfg.get('Ns', 1024)
        dummy = torch.randn(1, 1024, 3, device=device)

        if encoding_type == 'direct':
            encoded = direct_encode(dummy, T_infer)
        else:
            encoded = qsde_encode(dummy, Ns=Ns, T=T_infer)

        with torch.no_grad():
            with SynOpsCounter(model) as counter:
                _ = model(encoded)
            report = counter.get_report()

        run_name = ckpt_path.stem.replace('_best', '')
        energy_results.append({
            'run': run_name,
            'encoding': encoding_type,
            'T_infer': T_infer,
            'total_synops': report['total_synops'],
            'mean_spike_rate': report['mean_spike_rate'],
            'ac_energy_pj': report['ac_energy_pj'],
            'mac_energy_pj': report['mac_energy_pj'],
            'energy_ratio': report['energy_ratio'],
        })
        print(f"  {run_name}: SynOps={report['total_synops']:.3e}, "
              f"spike_rate={report['mean_spike_rate']:.4f}, "
              f"ratio={report['energy_ratio']:.1f}x")
else:
    print("No checkpoints found. Train a model first.")

In [ ]:
# Energy comparison bar chart
if energy_results:
    energy_df = pd.DataFrame(energy_results)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Energy Efficiency: SNN vs ANN Equivalent', fontsize=14, fontweight='bold')

    # AC vs MAC energy
    ax = axes[0]
    x = range(len(energy_df))
    width = 0.35
    bars1 = ax.bar([xi - width/2 for xi in x], energy_df['ac_energy_pj'],
                   width, label='SNN (AC)', color='#2196F3')
    bars2 = ax.bar([xi + width/2 for xi in x], energy_df['mac_energy_pj'],
                   width, label='ANN equiv. (MAC)', color='#FF5722')
    ax.set_xticks(list(x))
    ax.set_xticklabels(energy_df['run'], rotation=20, ha='right')
    ax.set_ylabel('Energy (pJ per inference)')
    ax.set_title('Energy: SNN vs ANN Equivalent')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    # Energy ratio
    ax = axes[1]
    bars = ax.bar(energy_df['run'], energy_df['energy_ratio'],
                  color='#4CAF50', edgecolor='white')
    for bar, ratio in zip(bars, energy_df['energy_ratio']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{ratio:.1f}x', ha='center', va='bottom', fontsize=12, fontweight='bold')
    ax.set_ylabel('Energy Ratio (ANN / SNN)')
    ax.set_title('Efficiency Advantage vs ANN')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    out_path = RESULTS_DIR / 'energy_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f"Saved to {out_path}")
    plt.show()
else:
    print("No energy data to plot.")

## 5. Confusion Matrix for Best Model

In [ ]:
# Load pre-saved confusion matrices if available
confusion_pngs = list(RESULTS_DIR.glob('confusion_*.png'))

if confusion_pngs:
    from IPython.display import Image, display
    for img_path in confusion_pngs:
        print(f"\n{img_path.name}:")
        display(Image(filename=str(img_path), width=700))
else:
    print("No confusion matrix PNGs found.")
    print("Run: python scripts/evaluate.py --checkpoint <path_to_best.pth>")

## 6. Summary Table

In [ ]:
# Summary table
rows = []

# Reference values
rows.append({'Model': 'ANN PointNet (ref)', 'Encoding': 'ReLU', 'Train T': '-',
             'Infer T': '-', 'Best OA (%)': 89.2, 'mAcc (%)': 86.0, 'Energy Ratio': '1x'})
rows.append({'Model': 'Spiking PointNet (paper)', 'Encoding': 'Direct', 'Train T': 1,
             'Infer T': 4, 'Best OA (%)': 88.61, 'mAcc (%)': '-', 'Energy Ratio': '~8x'})

# Our results
for run_name, df in dfs.items():
    best_oa = df['val_oa'].max() * 100
    best_macc = df.loc[df['val_oa'].idxmax(), 'val_macc'] * 100 if 'val_macc' in df.columns else '-'
    enc = 'Q-SDE' if 'qsde' in run_name.lower() else 'Direct'
    t_train = 4 if 'qsde' in run_name.lower() else 1

    energy_ratio = '-'
    for er in energy_results:
        if er['run'] == run_name:
            energy_ratio = f"{er['energy_ratio']:.1f}x"

    rows.append({
        'Model': f'Ours: {run_name}',
        'Encoding': enc,
        'Train T': t_train,
        'Infer T': 4,
        'Best OA (%)': round(best_oa, 2),
        'mAcc (%)': round(best_macc, 2) if isinstance(best_macc, float) else best_macc,
        'Energy Ratio': energy_ratio,
    })

summary_df = pd.DataFrame(rows)
print('\nResults Summary:')
print(summary_df.to_string(index=False))

# Display nicely
try:
    from IPython.display import HTML
    display(HTML(summary_df.to_html(index=False)))
except:
    pass